# Retry and fall back

**The job.** Fetch some data. The fast source is flaky. Use the slow one when it
breaks, and say so.

Two separate ideas here and they are easy to confuse:

* A **fallback** is another way to do the same step. Same contract, different
  implementation. The graph does not change.
* A **branch** is a different path through the work. The graph changes shape
  depending on what happened.

Both are here. Neither is a retry loop, and that is deliberate — a retry with no
limit is how a job runs forever.

In [ ]:
try:
    import browsergraph  # noqa: F401
except ImportError:
    %pip install -q "browsergraph @ git+https://github.com/aidonerightcorp/browsergraph.git"

import json, pathlib
from dataclasses import replace

from browsergraph import execute, viz
from browsergraph.compile import compile_route
from browsergraph.manifest import NodeManifest, PortSpec
from browsergraph.workbench import Edge, NodeCandidate, StageDefinition, WorkbenchDefinition

# A fresh folder each run. Left-over files from a previous run make the "what
# did this produce" list a lie, and that list is half the point here.
import shutil
WORK = pathlib.Path("work")
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir()

# These come from the library rather than being redefined in every notebook.
# They used to be thirty lines pasted into each one, which meant anyone copying
# a notebook to start a project got helpers that did not exist in browsergraph.
from browsergraph.quick import chain, fanin, fanout, link, node, problems, step
from browsergraph.quick import graph as _graph
from browsergraph.quick import subgraph  # noqa: F401  (used by later notebooks)

# The notebooks kept the older names, and `build` also prints what is wrong
# rather than raising — in a notebook the complaint is the lesson.
stage = step

def build(title, task, stages, nodes, edges=()):
    bench = _graph(title, task, stages, nodes, edges)
    print("problems:", problems(bench) or "none")
    return bench

print("ready")

## Two ways to get the data

`fetch.fast` fails about half the time. `fetch.slow` always works and takes
longer. Same ports, same contract, so either can fill the step.

In [ ]:
import random, time as clock
rng = random.Random(3)

attempts = {"fast": 0, "slow": 0, "cache": 0}

def fetch_fast(**kw):
    attempts["fast"] += 1
    if rng.random() < 0.6:
        raise ConnectionError("the fast source timed out")
    return {"source": "fast", "rows": [1, 2, 3, 4]}

def fetch_slow(**kw):
    attempts["slow"] += 1
    clock.sleep(0.01)
    return {"source": "slow", "rows": [1, 2, 3, 4]}

def fetch_cache(**kw):
    attempts["cache"] += 1
    return {"source": "cache (yesterday)", "rows": [1, 2, 3]}

print("three ways to do one step")

In [ ]:
nodes = [
    node("fetch.fast",  "fetch",  [], [("out", "Data")], runtime={"deterministic": False}),
    node("fetch.slow",  "fetch",  [], [("out", "Data")], runtime={"deterministic": False}),
    node("fetch.cache", "fetch",  [], [("out", "Data")]),
    node("grade.rows",  "grade",  [("in", "Data")], [("fresh", "Data"), ("stale", "Data")]),
    node("use.fresh",   "use",    [("in", "Data")], [("out", "Report")]),
    node("warn.stale",  "warn",   [("in", "Data")], [("out", "Report")]),
]

stages = [
    stage("fetch", "Get the data", [], [("out", "Data")], "fetch",
          ["fetch.fast", "fetch.slow", "fetch.cache"]),
    StageDefinition(id="grade", name="Fresh or stale?", kind="branch",
                    required_capabilities=("grade",),
                    inputs=(PortSpec("in", "Data"),),
                    outputs=(PortSpec("fresh", "Data"), PortSpec("stale", "Data")),
                    success="the data was graded",
                    candidates=("grade.rows",)),
    stage("use",  "Use it",       [("in", "Data")], [("out", "Report")], "use",  ["use.fresh"]),
    stage("warn", "Flag it",      [("in", "Data")], [("out", "Report")], "warn", ["warn.stale"]),
]

edges = [Edge("fetch", "grade"),
         Edge("grade", "use",  from_port="fresh"),
         Edge("grade", "warn", from_port="stale")]

bench = build("Fetch with a fallback",
              "Get the data from whichever source works, and say which one it was.",
              stages, nodes, edges)

In [ ]:
viz.dag(bench)

In [ ]:
def grade_rows(**kw):
    """Name the port. Cached data goes down the warning path."""
    data = kw["in"]
    return ("stale", data) if "cache" in data["source"] else ("fresh", data)

def use_fresh(**kw):
    return {"status": "used", "rows": len(kw["in"]["rows"]), "source": kw["in"]["source"]}

def warn_stale(**kw):
    return {"status": "used with a warning", "rows": len(kw["in"]["rows"]),
            "source": kw["in"]["source"], "warning": "this data is not from today"}

runtime = execute.Runtime({
    "fetch.fast": fetch_fast, "fetch.slow": fetch_slow, "fetch.cache": fetch_cache,
    "grade.rows": grade_rows, "use.fresh": use_fresh, "warn.stale": warn_stale,
})

route = {"fetch": "fetch.fast", "grade": "grade.rows",
         "use": "use.fresh", "warn": "warn.stale"}
plan = compile_route(bench, route)

## Run it ten times

Same plan every time. The fast source fails at random, so the fallbacks earn
their keep on some runs and not others.

In [ ]:
FALLBACKS = {"fetch": ["fetch.slow", "fetch.cache"]}

rows = []
for i in range(10):
    got = execute.run(plan, runtime, fallbacks=FALLBACKS)
    fetch_step = next(s for s in got.steps if s.stage == "fetch")
    taken = "use" if any(s.stage == "use" and not s.skipped for s in got.steps) else "warn"
    rows.append((i + 1, fetch_step.candidate, fetch_step.fell_back, taken, got.ok))

print(f"{'run':>4}  {'source used':<14}{'fell back':<11}{'path':<7}ok")
for i, source, fell, taken, ok in rows:
    print(f"{i:>4}  {source:<14}{str(fell):<11}{taken:<7}{ok}")

print(f"\nattempts: {attempts}")

The fast source was tried every single time. When it failed the slow one took
over, and the run still succeeded. Nothing in the graph changed — only which
candidate did the work, and the run says which one that was.

## What happens with no fallbacks

In [ ]:
bare = [execute.run(plan, runtime).ok for _ in range(10)]
print(f"without fallbacks: {sum(bare)}/10 runs succeeded")
print(f"with fallbacks:    {sum(1 for r in rows if r[4])}/10")

## The branch, when the data is old

Force the cache source and the graph takes the other path.

In [ ]:
forced = execute.run(compile_route(bench, dict(route, fetch="fetch.cache")), runtime)
print(forced.text())
print()
print("used path:  ", [s.stage for s in forced.steps if not s.skipped and s.stage in ("use", "warn")])
print("skipped:    ", [s.stage for s in forced.steps if s.skipped])
print("result:     ", forced.output("warn"))

The `use` step never ran, and it is recorded as **skipped** rather than failed.
A path not taken is a correct outcome. If it were logged as a failure every
branching run would look broken and nobody would read the logs.